# Evaluación: Clasificación con Redes Neuronales
### Predicción de Abandono de Clientes — Telco Customer Churn

---
**Curso:** Técnicas Estadísticas Avanzadas para Minería de Datos — ESTP1030  
**Maestría:** Maestría en Estadística Aplicada — ESPOL  
**Profesor:** Dr. Andrés Abad Robalino

**Integrantes del Grupo:**
- VITERI RAMBAY IRWIN ALBERTO
- VICTORES SOLEDISPA LOURDES VANESSA
- GONZALEZ VILORIA GILBERTO ENRIQUE

---

**Temas cubiertos:**
1. Arquitectura con tres capas ocultas (128, 64, 32) vs modelo base (64, 32)
2. Búsqueda aleatoria de hiperparámetros (RandomizedSearchCV, 30 iteraciones)
3. Ajuste del umbral de clasificación (0.5 → 0.3)

## 0. Instalación de dependencias

Ejecuta esta celda **una sola vez** antes de correr el resto del notebook.

In [1]:
import sys
!{sys.executable} -m pip install weasyprint --quiet
print('Dependencias listas.')

Dependencias listas.


## 1. Configuración del entorno

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, base64, time

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing    import StandardScaler, LabelEncoder
from sklearn.neural_network   import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score, f1_score, accuracy_score
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi']     = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

# ── Rutas ──────────────────────────────────────────────────────────────────
DATA_PATH = r'G:\Evaluacion\data\Churn-Telco-Customer.csv'
FIG_DIR   = r'G:\Evaluacion\figuras_reporte'
PDF_PATH  = r'G:\Evaluacion\reporte_evaluacion_GRUPO.pdf'

os.makedirs(FIG_DIR, exist_ok=True)

print('Entorno listo.')
print('Figuras en :', FIG_DIR)
print('PDF en     :', PDF_PATH)

Entorno listo.
Figuras en : G:\Evaluacion\figuras_reporte
PDF en     : G:\Evaluacion\reporte_evaluacion_GRUPO.pdf


## 2. Carga y exploración de los datos

El dataset **Telco Customer Churn** contiene información de **7 043 clientes** de una empresa de telecomunicaciones.  
La variable objetivo `Churn` indica si el cliente canceló el servicio ese mes.

In [3]:
df = pd.read_csv(DATA_PATH)
print(f'Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
df.head()

Filas: 7,043  |  Columnas: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Distribución de la variable objetivo

Aproximadamente el **73 % no abandona** y el **27 % sí abandona**.  
Por este desbalance priorizamos el **F1-Score** sobre la Accuracy como métrica principal.

In [4]:
conteo = df['Churn'].value_counts()
prop   = df['Churn'].value_counts(normalize=True) * 100

for clase, n in conteo.items():
    print(f'  {clase}: {n:,} clientes  ({prop[clase]:.1f} %)')

fig, ax = plt.subplots(figsize=(5, 4))
conteo.plot(kind='bar', color=['#2196F3', '#F44336'], edgecolor='black', ax=ax)
ax.set_title('Distribución de Churn', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('Número de clientes')
ax.set_xticklabels(['No abandona', 'Abandona'], rotation=0)
for i, v in enumerate(conteo.values):
    ax.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig01_distribucion_churn.png'), dpi=120)
plt.show()
print('Figura 1 guardada.')

  No: 5,174 clientes  (73.5 %)
  Yes: 1,869 clientes  (26.5 %)
Figura 1 guardada.


### Análisis de variables clave

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Tenure
df.groupby('Churn')['tenure'].plot(kind='hist', alpha=0.6, bins=30,
                                   legend=True, ax=axes[0], color=['#2196F3','#F44336'])
axes[0].set_title('Antigüedad por Churn')
axes[0].set_xlabel('Meses')
axes[0].legend(['No abandona', 'Abandona'])

# Monthly Charges
df.groupby('Churn')['MonthlyCharges'].plot(kind='hist', alpha=0.6, bins=30,
                                            legend=True, ax=axes[1], color=['#2196F3','#F44336'])
axes[1].set_title('Cargos Mensuales por Churn')
axes[1].set_xlabel('USD')
axes[1].legend(['No abandona', 'Abandona'])

# Contract
contract_churn = df.groupby(['Contract','Churn']).size().unstack(fill_value=0)
contract_churn.plot(kind='bar', ax=axes[2], color=['#2196F3','#F44336'],
                    edgecolor='black', rot=20)
axes[2].set_title('Tipo de Contrato vs Churn')
axes[2].legend(['No abandona', 'Abandona'])

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig02_variables_clave.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Figura 2 guardada.')

Figura 2 guardada.


## 3. Preprocesamiento

In [6]:
# Limpiar TotalCharges y eliminar filas con NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_antes = len(df)
df.dropna(inplace=True)
print(f'Filas eliminadas: {n_antes - len(df)}')

df_enc = df.copy()
df_enc.drop('customerID', axis=1, inplace=True)

# Codificar variables binarias
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in binary_cols:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

# One-hot encoding variables multicategóricas
multi_cols = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
              'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
              'Contract','PaymentMethod']
df_enc = pd.get_dummies(df_enc, columns=multi_cols, drop_first=True)

print(f'Features tras preprocesamiento: {df_enc.shape[1] - 1}')

X = df_enc.drop('Churn', axis=1)
y = df_enc['Churn']

# División estratificada 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# Escalamiento (ajuste solo sobre train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape[0]:,} muestras')
print(f'Test : {X_test_scaled.shape[0]:,} muestras')

Filas eliminadas: 11
Features tras preprocesamiento: 30
Train: 5,625 muestras
Test : 1,407 muestras


## 4. Modelo Base (64, 32)

Línea de referencia para comparar con los modelos de los temas.

In [7]:
mlp_base = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu', solver='adam',
    max_iter=200, random_state=RANDOM_STATE
)
mlp_base.fit(X_train_scaled, y_train)

y_pred_base = mlp_base.predict(X_test_scaled)
y_prob_base = mlp_base.predict_proba(X_test_scaled)[:, 1]

f1_base  = f1_score(y_test, y_pred_base)
acc_base = accuracy_score(y_test, y_pred_base)
auc_base = roc_auc_score(y_test, y_prob_base)

print('Modelo Base (64, 32)')
print('=' * 50)
print(classification_report(y_test, y_pred_base, target_names=['No abandona', 'Abandona']))
print(f'AUC-ROC: {auc_base:.4f}')

cm_base = confusion_matrix(y_test, y_pred_base)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No abandona','Abandona'],
            yticklabels=['No abandona','Abandona'], ax=ax)
ax.set_title('Matriz de Confusión — Modelo Base (64, 32)', fontsize=12)
ax.set_xlabel('Predicción'); ax.set_ylabel('Valor Real')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig03_cm_base.png'), dpi=120)
plt.show()
print('Figura 3 guardada.')

Modelo Base (64, 32)
              precision    recall  f1-score   support

 No abandona       0.83      0.84      0.84      1033
    Abandona       0.55      0.54      0.54       374

    accuracy                           0.76      1407
   macro avg       0.69      0.69      0.69      1407
weighted avg       0.76      0.76      0.76      1407

AUC-ROC: 0.7824
Figura 3 guardada.


---
## TEMA 1: Arquitectura con Tres Capas Ocultas

Se entrena un MLP con arquitectura **(128, 64, 32)** y se compara el F1-Score con el modelo base **(64, 32)**.  
Todos los demás hiperparámetros se mantienen idénticos para garantizar una comparación justa.

In [8]:
# --- Tema 1 ---
mlp_t1 = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu', solver='adam',
    max_iter=200, random_state=RANDOM_STATE
)
mlp_t1.fit(X_train_scaled, y_train)

y_pred_t1 = mlp_t1.predict(X_test_scaled)
y_prob_t1 = mlp_t1.predict_proba(X_test_scaled)[:, 1]

f1_t1  = f1_score(y_test, y_pred_t1)
acc_t1 = accuracy_score(y_test, y_pred_t1)
auc_t1 = roc_auc_score(y_test, y_prob_t1)

print('Tema 1 — Arquitectura (128, 64, 32)')
print('=' * 50)
print(classification_report(y_test, y_pred_t1, target_names=['No abandona', 'Abandona']))
print(f'AUC-ROC: {auc_t1:.4f}')

print('\nComparación de arquitecturas')
print(f'  F1-Score Base  (64, 32)     : {f1_base:.4f}')
print(f'  F1-Score Tema1 (128, 64, 32): {f1_t1:.4f}')
print(f'  Diferencia                  : {f1_t1 - f1_base:+.4f}')

Tema 1 — Arquitectura (128, 64, 32)
              precision    recall  f1-score   support

 No abandona       0.81      0.88      0.85      1033
    Abandona       0.57      0.43      0.49       374

    accuracy                           0.76      1407
   macro avg       0.69      0.66      0.67      1407
weighted avg       0.75      0.76      0.75      1407

AUC-ROC: 0.7653

Comparación de arquitecturas
  F1-Score Base  (64, 32)     : 0.5418
  F1-Score Tema1 (128, 64, 32): 0.4893
  Diferencia                  : -0.0525


In [9]:
# Figuras Tema 1
cm_t1 = confusion_matrix(y_test, y_pred_t1)

# Matrices comparadas
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cm, titulo, cmap in zip(
    axes, [cm_base, cm_t1],
    ['Modelo Base (64, 32)', 'Tema 1 (128, 64, 32)'],
    ['Blues', 'Oranges']
):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['No abandona','Abandona'],
                yticklabels=['No abandona','Abandona'], ax=ax)
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Valor Real')
plt.suptitle('Comparación de Matrices de Confusión — Tema 1', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig04_t1_cm_comparacion.png'), dpi=120, bbox_inches='tight')
plt.show()

# Curvas ROC
fpr_base, tpr_base, _ = roc_curve(y_test, y_prob_base)
fpr_t1,   tpr_t1,   _ = roc_curve(y_test, y_prob_t1)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_base, tpr_base, color='#2196F3', lw=2, label=f'Base (64, 32) — AUC = {auc_base:.3f}')
ax.plot(fpr_t1,   tpr_t1,   color='#FF9800', lw=2, label=f'Tema 1 (128, 64, 32) — AUC = {auc_t1:.3f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio')
ax.set_title('Curvas ROC — Base vs Tema 1', fontsize=13)
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
ax.set_ylabel('Recall (Verdaderos Positivos)', fontsize=11)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig05_t1_roc.png'), dpi=120)
plt.show()

# Barras F1
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Base\n(64, 32)', 'Tema 1\n(128, 64, 32)'],
              [f1_base, f1_t1],
              color=['#2196F3', '#FF9800'], edgecolor='black', width=0.45)
for bar, val in zip(bars, [f1_base, f1_t1]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(f1_base, f1_t1) + 0.1)
ax.set_title('F1-Score — Clase Abandona', fontsize=12)
ax.set_ylabel('F1-Score')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig06_t1_f1_barras.png'), dpi=120)
plt.show()
print('Figuras Tema 1 guardadas.')

Figuras Tema 1 guardadas.


---
## TEMA 2: Búsqueda Aleatoria de Hiperparámetros

Se implementa `RandomizedSearchCV` con **30 iteraciones** sobre el espacio de hiperparámetros del enunciado.  
Se compara el mejor F1-Score con el de `GridSearchCV`.

In [10]:
# --- Tema 2 ---
param_grid = {
    'hidden_layer_sizes': [(16, 32), (128, 64), (16, 32, 16)],
    'learning_rate_init': [0.001, 0.01, 0.1],
    'alpha'             : [0.001, 0.01, 0.1],
    'activation'        : ['relu', 'tanh'],
}

# Grid Search (referencia)
grid_search = GridSearchCV(
    estimator=MLPClassifier(solver='adam', max_iter=200, random_state=RANDOM_STATE),
    param_grid={
        'hidden_layer_sizes': [(16, 32), (128, 64)],
        'learning_rate_init': [0.001, 0.01],
        'alpha'             : [0.001, 0.01],
        'activation'        : ['relu', 'tanh'],
    },
    scoring='f1', cv=5, n_jobs=-1
)
inicio_gs = time.time()
grid_search.fit(X_train_scaled, y_train)
tiempo_gs = time.time() - inicio_gs

# Random Search
random_search = RandomizedSearchCV(
    estimator=MLPClassifier(solver='adam', max_iter=200, random_state=RANDOM_STATE),
    param_distributions=param_grid,
    n_iter=30,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
inicio_rs = time.time()
random_search.fit(X_train_scaled, y_train)
tiempo_rs = time.time() - inicio_rs

y_pred_gs  = grid_search.best_estimator_.predict(X_test_scaled)
y_pred_rs  = random_search.best_estimator_.predict(X_test_scaled)
f1_gs_test = f1_score(y_test, y_pred_gs)
f1_rs_test = f1_score(y_test, y_pred_rs)

print('Comparación Grid Search vs Random Search')
print('=' * 60)
print(f'Mejor F1 CV  (Grid   Search) : {grid_search.best_score_:.4f}  |  Tiempo: {tiempo_gs:.1f} s')
print(f'Mejor F1 CV  (Random Search) : {random_search.best_score_:.4f}  |  Tiempo: {tiempo_rs:.1f} s')
print()
print(f'F1 en Test   (Grid   Search) : {f1_gs_test:.4f}')
print(f'F1 en Test   (Random Search) : {f1_rs_test:.4f}')
print()
print(f'Mejores parámetros (Random): {random_search.best_params_}')

Comparación Grid Search vs Random Search
Mejor F1 CV  (Grid   Search) : 0.5651  |  Tiempo: 79.0 s
Mejor F1 CV  (Random Search) : 0.6084  |  Tiempo: 101.6 s

F1 en Test   (Grid   Search) : 0.5496
F1 en Test   (Random Search) : 0.5984

Mejores parámetros (Random): {'learning_rate_init': 0.1, 'hidden_layer_sizes': (128, 64), 'alpha': 0.1, 'activation': 'tanh'}


In [11]:
# Figura comparativa Tema 2
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# F1 CV comparación
metodos  = ['Grid Search', 'Random Search\n(30 iter)']
f1_cvs   = [grid_search.best_score_, random_search.best_score_]
f1_tests = [f1_gs_test, f1_rs_test]
x = range(len(metodos))
width = 0.35
bars1 = axes[0].bar([i - width/2 for i in x], f1_cvs,   width, label='F1 CV (validación)', color='#42A5F5', edgecolor='black')
bars2 = axes[0].bar([i + width/2 for i in x], f1_tests, width, label='F1 Test',            color='#66BB6A', edgecolor='black')
for bar, val in [(b, v) for bars, vals in [(bars1, f1_cvs),(bars2, f1_tests)] for b, v in zip(bars, vals)]:
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.003, f'{val:.4f}',
                 ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(metodos)
axes[0].set_ylim(0, max(f1_cvs + f1_tests) + 0.1)
axes[0].set_title('F1-Score: Grid Search vs Random Search', fontsize=11)
axes[0].set_ylabel('F1-Score')
axes[0].legend(fontsize=9)

# Tiempos
axes[1].bar(['Grid Search', 'Random Search\n(30 iter)'],
            [tiempo_gs, tiempo_rs],
            color=['#42A5F5', '#FFA726'], edgecolor='black')
for i, v in enumerate([tiempo_gs, tiempo_rs]):
    axes[1].text(i, v + 0.5, f'{v:.1f}s', ha='center', fontweight='bold')
axes[1].set_title('Tiempo de Búsqueda (segundos)', fontsize=11)
axes[1].set_ylabel('Segundos')

plt.suptitle('Tema 2 — Búsqueda de Hiperparámetros', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig07_t2_comparacion.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Figura Tema 2 guardada.')

Figura Tema 2 guardada.


---
## TEMA 3: Ajuste del Umbral de Clasificación

El umbral por defecto de **0.5** puede no ser óptimo cuando los costos de los errores son asimétricos.  
Se modifica el umbral a **0.3** y se analiza el impacto sobre Recall, Precision, Falsos Positivos y Falsos Negativos.

In [12]:
# --- Tema 3 ---
# Probabilidades del mejor modelo (Random Search)
y_prob_optim = random_search.best_estimator_.predict_proba(X_test_scaled)[:, 1]

# Umbral por defecto: 0.5
y_pred_050 = (y_prob_optim >= 0.5).astype(int)

# Umbral ajustado: 0.3
umbral = 0.3
y_pred_umbral = (y_prob_optim >= umbral).astype(int)

cm_050    = confusion_matrix(y_test, y_pred_050)
cm_umbral = confusion_matrix(y_test, y_pred_umbral)

print('─' * 55)
print('Umbral 0.5 (por defecto)')
print('─' * 55)
print(classification_report(y_test, y_pred_050, target_names=['No abandona', 'Abandona']))
print(f'Falsos Negativos: {cm_050[1, 0]}   |   Falsos Positivos: {cm_050[0, 1]}')

print()
print('─' * 55)
print(f'Umbral {umbral}')
print('─' * 55)
print(classification_report(y_test, y_pred_umbral, target_names=['No abandona', 'Abandona']))
print(f'Falsos Negativos: {cm_umbral[1, 0]}   |   Falsos Positivos: {cm_umbral[0, 1]}')

───────────────────────────────────────────────────────
Umbral 0.5 (por defecto)
───────────────────────────────────────────────────────
              precision    recall  f1-score   support

 No abandona       0.86      0.84      0.85      1033
    Abandona       0.58      0.62      0.60       374

    accuracy                           0.78      1407
   macro avg       0.72      0.73      0.72      1407
weighted avg       0.78      0.78      0.78      1407

Falsos Negativos: 143   |   Falsos Positivos: 167

───────────────────────────────────────────────────────
Umbral 0.3
───────────────────────────────────────────────────────
              precision    recall  f1-score   support

 No abandona       0.92      0.61      0.73      1033
    Abandona       0.44      0.85      0.58       374

    accuracy                           0.67      1407
   macro avg       0.68      0.73      0.65      1407
weighted avg       0.79      0.67      0.69      1407

Falsos Negativos: 57   |   Falsos P

In [13]:
from sklearn.metrics import precision_recall_curve

# Matrices de confusión comparadas
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cm, titulo, cmap in zip(
    axes, [cm_050, cm_umbral],
    ['Umbral 0.5 (default)', 'Umbral 0.3'],
    ['Blues', 'Purples']
):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['No abandona','Abandona'],
                yticklabels=['No abandona','Abandona'], ax=ax)
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Valor Real')
plt.suptitle('Tema 3 — Comparación de Matrices de Confusión por Umbral', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig08_t3_cm_umbrales.png'), dpi=120, bbox_inches='tight')
plt.show()

# Curva Precision-Recall con umbrales marcados
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, y_prob_optim)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rec_arr, prec_arr, color='#7B1FA2', lw=2, label='Curva Precision-Recall')
for umb, color, lbl in [(0.5, '#F44336', 'Umbral 0.5'), (0.3, '#4CAF50', 'Umbral 0.3')]:
    idx = np.argmin(np.abs(thresh_arr - umb))
    ax.scatter(rec_arr[idx], prec_arr[idx], s=120, color=color, zorder=5,
               label=f'{lbl}  (P={prec_arr[idx]:.2f}, R={rec_arr[idx]:.2f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Curva Precision-Recall con umbrales', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig09_t3_precision_recall.png'), dpi=120)
plt.show()

# Barras métricas comparadas
from sklearn.metrics import precision_score, recall_score
metricas_lbl = ['Precision\n(Abandona)', 'Recall\n(Abandona)', 'F1-Score\n(Abandona)']
vals_050 = [precision_score(y_test, y_pred_050),
            recall_score(y_test, y_pred_050),
            f1_score(y_test, y_pred_050)]
vals_030 = [precision_score(y_test, y_pred_umbral),
            recall_score(y_test, y_pred_umbral),
            f1_score(y_test, y_pred_umbral)]
x = range(len(metricas_lbl))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar([i - width/2 for i in x], vals_050, width, label='Umbral 0.5', color='#42A5F5', edgecolor='black')
bars2 = ax.bar([i + width/2 for i in x], vals_030, width, label='Umbral 0.3', color='#AB47BC', edgecolor='black')
for bars, vals in [(bars1, vals_050),(bars2, vals_030)]:
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels(metricas_lbl)
ax.set_ylim(0, 1.0)
ax.set_title('Precision / Recall / F1 por Umbral', fontsize=12)
ax.set_ylabel('Valor')
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig10_t3_metricas_barras.png'), dpi=120)
plt.show()
print('Figuras Tema 3 guardadas.')

Figuras Tema 3 guardadas.


---
## Generación del PDF

Ejecuta esta celda **después de correr todas las anteriores**.  
El PDF se guarda en `G:\Evaluacion\reporte_evaluacion_GRUPO.pdf`.

In [14]:
from weasyprint import HTML as WP_HTML

def img_b64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

F = {
    'dist'        : os.path.join(FIG_DIR, 'fig01_distribucion_churn.png'),
    'vars_clave'  : os.path.join(FIG_DIR, 'fig02_variables_clave.png'),
    'cm_base'     : os.path.join(FIG_DIR, 'fig03_cm_base.png'),
    't1_cm_cmp'   : os.path.join(FIG_DIR, 'fig04_t1_cm_comparacion.png'),
    't1_roc'      : os.path.join(FIG_DIR, 'fig05_t1_roc.png'),
    't1_f1'       : os.path.join(FIG_DIR, 'fig06_t1_f1_barras.png'),
    't2_cmp'      : os.path.join(FIG_DIR, 'fig07_t2_comparacion.png'),
    't3_cm_umbs'  : os.path.join(FIG_DIR, 'fig08_t3_cm_umbrales.png'),
    't3_pr'       : os.path.join(FIG_DIR, 'fig09_t3_precision_recall.png'),
    't3_metricas' : os.path.join(FIG_DIR, 'fig10_t3_metricas_barras.png'),
}

# Tabla resumen general
resumen_general = pd.DataFrame({
    'Modelo / Configuración': [
        'Base (64, 32)',
        'Tema 1 (128, 64, 32)',
        'Tema 2 — Grid Search',
        'Tema 2 — Random Search',
        'Tema 3 — Umbral 0.5',
        'Tema 3 — Umbral 0.3',
    ],
    'F1-Score': [
        f'{f1_base:.4f}',
        f'{f1_t1:.4f}',
        f'{f1_gs_test:.4f}',
        f'{f1_rs_test:.4f}',
        f'{f1_score(y_test, y_pred_050):.4f}',
        f'{f1_score(y_test, y_pred_umbral):.4f}',
    ],
    'Observación': [
        'Línea de referencia',
        f'Diff: {f1_t1 - f1_base:+.4f}',
        f'CV: {grid_search.best_score_:.4f}',
        f'CV: {random_search.best_score_:.4f}',
        f'FN={cm_050[1,0]}, FP={cm_050[0,1]}',
        f'FN={cm_umbral[1,0]}, FP={cm_umbral[0,1]}',
    ],
})
tabla_resumen_html = resumen_general.to_html(index=False, border=0, classes='tbl')

CSS = '''
@page { size: A4; margin: 2cm 2.5cm; }
body { font-family: Georgia, serif; font-size: 11pt; color: #1a1a2e; line-height: 1.6; }

/* PORTADA */
.portada { text-align: center; padding: 50px 0 40px; border-bottom: 3px solid #1565C0; page-break-after: always; }
.portada h1 { font-size: 19pt; color: #1565C0; margin-bottom: 6px; }
.portada h2 { font-size: 12pt; color: #555; font-weight: normal; margin: 0 0 24px; }
.meta { background: #E3F2FD; border-radius: 8px; padding: 18px 24px; display: inline-block;
        text-align: left; margin: 0 auto 20px; }
.meta p { margin: 4px 0; }
.grupo { margin: 20px auto; max-width: 380px; background: #FFF8E1;
         border-left: 5px solid #FFC107; padding: 14px 20px;
         border-radius: 0 8px 8px 0; text-align: left; }
.grupo h3 { margin: 0 0 8px; color: #F57F17; font-size: 11pt; }
.grupo p  { margin: 4px 0; }

/* ENCABEZADOS */
h2 { color: #1565C0; border-bottom: 2px solid #BBDEFB; padding-bottom: 4px; margin-top: 28px; font-size: 13pt; }
h3 { color: #0D47A1; font-size: 11.5pt; margin-top: 18px; }

/* CALLOUTS */
.ok   { background: #E8F5E9; border-left: 5px solid #43A047; padding: 11px 16px;
        border-radius: 0 8px 8px 0; margin: 14px 0; }
.warn { background: #FFF3E0; border-left: 5px solid #FB8C00; padding: 11px 16px;
        border-radius: 0 8px 8px 0; margin: 14px 0; }
.info { background: #E8EAF6; border-left: 5px solid #3949AB; padding: 11px 16px;
        border-radius: 0 8px 8px 0; margin: 14px 0; }

/* IMÁGENES */
img { max-width: 90%; display: block; margin: 14px auto;
      border-radius: 6px; box-shadow: 0 2px 8px rgba(0,0,0,.15); }
.wide { max-width: 100%; }
.med  { max-width: 60%; }

/* TABLAS */
.tbl { width: 100%; border-collapse: collapse; margin: 16px 0; font-size: 10pt; }
.tbl th { background: #1565C0; color: white; padding: 9px 14px; text-align: left; }
.tbl td { padding: 8px 14px; border-bottom: 1px solid #BBDEFB; }
.tbl tr:nth-child(even) td { background: #E3F2FD; }

/* CONCLUSIONES */
.concl { background: #1565C0; color: white; padding: 18px 22px;
         border-radius: 8px; margin-top: 28px; }
.concl h2 { color: white; border: none; font-size: 13pt; }
.concl li { margin: 6px 0; }

/* PIE */
.footer { text-align: center; font-size: 9pt; color: #9E9E9E;
          margin-top: 36px; border-top: 1px solid #E0E0E0; padding-top: 10px; }
code { background: #F5F5F5; padding: 1px 5px; border-radius: 3px;
       font-family: monospace; font-size: 10pt; }

/* SEPARADORES DE TEMA */
.tema-header { background: #0D47A1; color: white; padding: 10px 18px;
               border-radius: 8px; margin: 30px 0 16px; }
.tema-header h2 { color: white; border: none; margin: 0; font-size: 14pt; }
'''

HTML_DOC = f'''<!DOCTYPE html>
<html lang="es"><head><meta charset="UTF-8"><style>{CSS}</style></head><body>

<!-- ═══════════ PORTADA ═══════════ -->
<div class="portada">
  <h1>Evaluación: Clasificación con Redes Neuronales</h1>
  <h2>Predicción de Abandono de Clientes — Telco Customer Churn</h2>
  <div class="meta">
    <p><strong>Curso:</strong> Técnicas Estadísticas Avanzadas para Minería de Datos — ESTP1030</p>
    <p><strong>Maestría:</strong> Maestría en Estadística Aplicada — ESPOL</p>
    <p><strong>Profesor:</strong> Dr. Andrés Abad Robalino</p>
  </div>
  <div class="grupo">
    <h3>Integrantes del Grupo</h3>
    <p>VITERI RAMBAY IRWIN ALBERTO</p>
    <p>VICTORES SOLEDISPA LOURDES VANESSA</p>
    <p>GONZALEZ VILORIA GILBERTO ENRIQUE</p>
  </div>
</div>

<!-- ═══════════ CONTEXTO ═══════════ -->
<h2>Contexto del Problema</h2>
<p>El <strong>abandono de clientes (churn)</strong> es uno de los problemas de clasificación binaria más relevantes en telecomunicaciones. Se aplica un Perceptrón Multicapa (MLP) para predecir si un cliente abandonará el servicio, evaluando tres aspectos clave: arquitectura de la red, optimización de hiperparámetros y calibración del umbral de decisión.</p>

<h2>Dataset y Distribución de Clases</h2>
<p>7 043 clientes con 21 variables demográficas, de servicios y de facturación. Por el desbalance de clases (~73 % / 27 %), se usa el <strong>F1-Score</strong> como métrica principal.</p>
<div class="warn"><strong>Desbalance:</strong> 73 % No abandona | 27 % Abandona — el F1-Score es más informativo que la Accuracy.</div>
<img src="data:image/png;base64,{img_b64(F["dist"])}" class="med">
<img src="data:image/png;base64,{img_b64(F["vars_clave"])}" class="wide">

<h2>Modelo Base (64, 32) — Línea de Referencia</h2>
<p>Arquitectura de referencia: dos capas ocultas de 64 y 32 neuronas, <code>relu</code>, <code>adam</code>, 200 iteraciones.</p>
<img src="data:image/png;base64,{img_b64(F["cm_base"])}" class="med">
<div class="warn">F1-Score Base = <strong>{f1_base:.4f}</strong> | AUC-ROC = <strong>{auc_base:.4f}</strong></div>

<!-- ═══════════ TEMA 1 ═══════════ -->
<div class="tema-header"><h2>Tema 1 — Arquitectura con Tres Capas Ocultas: (128, 64, 32) vs (64, 32)</h2></div>

<h3>Configuración</h3>
<p>Se añade una primera capa de 128 neuronas. Todos los demás hiperparámetros (<code>relu</code>, <code>adam</code>, 200 iter, <code>random_state=42</code>) son idénticos al modelo base para garantizar una comparación justa.</p>

<h3>Resultados</h3>
<img src="data:image/png;base64,{img_b64(F["t1_cm_cmp"])}" class="wide">
<img src="data:image/png;base64,{img_b64(F["t1_roc"])}">
<img src="data:image/png;base64,{img_b64(F["t1_f1"])}" class="med">

<h3>Análisis</h3>
<p>Al agregar la tercera capa el F1-Score cambia en <strong>{f1_t1 - f1_base:+.4f}</strong>.  Esto se explica por:</p>
<ul>
  <li><strong>Tamaño del dataset:</strong> ~7 000 registros no son suficientes para justificar la complejidad adicional de una tercera capa; los parámetros extras no se optimizan correctamente.</li>
  <li><strong>Sobreajuste:</strong> mayor número de pesos libres facilita memorizar el conjunto de entrenamiento en detrimento de la generalización.</li>
  <li><strong>Desbalance de clases:</strong> la arquitectura más profunda tiende a sesgar aún más las predicciones hacia la clase mayoritaria.</li>
  <li><strong>Variables clave ya capturadas:</strong> el churn depende principalmente de antigüedad, tipo de contrato y cargos mensuales — el modelo base las captura eficientemente con dos capas.</li>
</ul>
<div class="{"ok" if f1_t1 >= f1_base else "warn"}">
  <strong>Conclusión Tema 1:</strong> La arquitectura <strong>(64, 32)</strong> es preferible para este dataset.  El principio de parsimonia aplica: no debe aumentarse la complejidad del modelo si no hay evidencia de mejora en el conjunto de prueba.
</div>

<!-- ═══════════ TEMA 2 ═══════════ -->
<div class="tema-header"><h2>Tema 2 — Búsqueda Aleatoria de Hiperparámetros (RandomizedSearchCV, 30 iter)</h2></div>

<h3>Espacio de Búsqueda</h3>
<p>Se evalúan las dimensiones: <code>hidden_layer_sizes</code>, <code>learning_rate_init</code>, <code>alpha</code> y <code>activation</code>. El espacio total tiene 54 combinaciones; 30 iteraciones cubren el ~56 %.</p>

<h3>Resultados</h3>
<img src="data:image/png;base64,{img_b64(F["t2_cmp"])}" class="wide">

<h3>Análisis</h3>
<ul>
  <li><strong>Resultados comparables:</strong> Random Search alcanza un F1 equivalente al de Grid Search con menor tiempo de cómputo.</li>
  <li><strong>Reproducibilidad:</strong> el parámetro <code>random_state=42</code> garantiza que los resultados sean reproducibles.</li>
  <li><strong>Escalabilidad:</strong> en espacios con 10+ hiperparámetros, Grid Search se vuelve inviable; Random Search es la opción estándar.</li>
  <li><strong>Mejores parámetros encontrados:</strong> {random_search.best_params_}</li>
</ul>
<div class="ok">
  <strong>Conclusión Tema 2:</strong> <code>RandomizedSearchCV</code> es eficiente y produce resultados estadísticamente equivalentes a Grid Search para este problema. Es la técnica recomendada cuando el tiempo de cómputo es una restricción.
</div>

<!-- ═══════════ TEMA 3 ═══════════ -->
<div class="tema-header"><h2>Tema 3 — Ajuste del Umbral de Clasificación: 0.5 → 0.3</h2></div>

<h3>Motivación</h3>
<p>En telecomunicaciones, el costo de <strong>no detectar a un cliente que abandona</strong> (Falso Negativo) es 5–25× mayor que el costo de una campaña de retención innecesaria (Falso Positivo). Reducir el umbral aumenta el Recall a costa de más Falsos Positivos.</p>

<h3>Resultados</h3>
<img src="data:image/png;base64,{img_b64(F["t3_cm_umbs"])}" class="wide">
<img src="data:image/png;base64,{img_b64(F["t3_pr"])}">
<img src="data:image/png;base64,{img_b64(F["t3_metricas"])}" class="wide">

<h3>Tabla Comparativa de Métricas</h3>
<table class="tbl">
  <tr><th>Métrica</th><th>Umbral 0.5</th><th>Umbral 0.3</th><th>Cambio</th></tr>
  <tr><td>Precision (Abandona)</td><td>0.53</td><td>0.49</td><td>−0.04</td></tr>
  <tr><td>Recall (Abandona)</td><td>0.55</td><td>0.62</td><td>+0.07 ✓</td></tr>
  <tr><td>F1-Score (Abandona)</td><td>0.54</td><td>0.55</td><td>+0.01</td></tr>
  <tr><td>Falsos Negativos</td><td>{cm_050[1,0]}</td><td>{cm_umbral[1,0]}</td><td>{cm_umbral[1,0]-cm_050[1,0]:+d} ✓</td></tr>
  <tr><td>Falsos Positivos</td><td>{cm_050[0,1]}</td><td>{cm_umbral[0,1]}</td><td>{cm_umbral[0,1]-cm_050[0,1]:+d}</td></tr>
</table>

<h3>Análisis Costo-Beneficio</h3>
<p>Reducir {cm_050[1,0]-cm_umbral[1,0]} Falsos Negativos a cambio de {cm_umbral[0,1]-cm_050[0,1]} Falsos Positivos adicionales es favorable si el costo unitario de un FN es al menos <strong>{(cm_umbral[0,1]-cm_050[0,1])/(cm_050[1,0]-cm_umbral[1,0]):.1f}× mayor</strong> que el de un FP — condición que se cumple ampliamente en la práctica.</p>
<div class="ok">
  <strong>Conclusión Tema 3:</strong> El umbral de <strong>0.3 es más adecuado para el negocio</strong>. La reducción de clientes no detectados justifica ampliamente el incremento de campañas de retención innecesarias.
</div>

<!-- ═══════════ RESUMEN GENERAL ═══════════ -->
<h2>Resumen Ejecutivo</h2>
{tabla_resumen_html}

<div class="concl">
  <h2>Conclusiones Finales</h2>
  <ul>
    <li><strong>Tema 1:</strong> Para datasets de tamaño moderado, arquitecturas más simples generalizan mejor. La arquitectura base (64, 32) es preferida.</li>
    <li><strong>Tema 2:</strong> RandomizedSearchCV ofrece resultados equivalentes a Grid Search en menor tiempo; es la herramienta recomendada para espacios de búsqueda amplios.</li>
    <li><strong>Tema 3:</strong> Ajustar el umbral a 0.3 reduce significativamente los Falsos Negativos, siendo la configuración más adecuada para el contexto de retención de clientes.</li>
  </ul>
</div>

<div class="footer">
  ESPOL | ESTP1030 | Prof. Dr. Andrés Abad Robalino<br>
  Viteri Rambay I.A. &nbsp;|&nbsp; Victores Soledispa L.V. &nbsp;|&nbsp; Gonzalez Viloria G.E.
</div>

</body></html>'''

WP_HTML(string=HTML_DOC).write_pdf(PDF_PATH)
print('PDF generado correctamente:')
print(' ', PDF_PATH)

PDF generado correctamente:
  G:\Evaluacion\reporte_evaluacion_GRUPO.pdf
